<a href="https://colab.research.google.com/github/lydiacyhung/114-2-ProgramingLanguage/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-generativeai

In [3]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json
import re # Add import re for regular expressions

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [39]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('Gemini-1')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [33]:
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=1012211887#gid=1012211887'

REQUIRED_COLUMNS = ["學年度", "學期", "科目", "成績"]

_auth_done = False
_gc = None
_ws = None

In [34]:
# --- 輸入成績資料 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        year = input(f"請輸入學年度")
        try:
            year = int(year)
        except ValueError:
            print("學年度必須是數字。請重新輸入。")
            continue

        semester = input(f"請輸入學期")
        try:
            semester = int(semester)
        except ValueError:
            print("學期必須是數字。請重新輸入。")
            continue
        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([year, semester, subject, grade])
        print(f"已記錄：學年度: {year}, 學期: {semester}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [35]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        year, semester, subject, grade = record
        prompt_text += f"學期：{year}-{semester}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [36]:

new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：英文
請輸入學年度13
請輸入學期1
請輸入 英文 的成績：q
成績必須是數字。請重新輸入。
請輸入科目（或輸入 'q' 停止）：英文
請輸入學年度113
請輸入學期1
請輸入 英文 的成績：80
已記錄：學年度: 113, 學期: 1, 科目: 英文, 成績: 80

請輸入科目（或輸入 'q' 停止）：q


In [37]:
new_grades

[[113, 1, '英文', 80]]

In [41]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Convert new_grades list to a DataFrame
df_grades = pd.DataFrame(new_grades, columns=['學年度', '學期', '科目', '成績'])

# Combine '學年度' and '學期' for a chronological order
df_grades['學期序'] = df_grades['學年度'].astype(str) + '-' + df_grades['學期'].astype(str)

# Sort by academic period
df_grades = df_grades.sort_values(by=['學年度', '學期'])

print("--- 成績資料 DataFrame --- ")
display(df_grades)

--- 成績資料 DataFrame --- 


,學年度,學期,科目,成績,學期序
0,113,1,英文,80,113-1


In [40]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---
呼叫 AI 時發生錯誤：503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


'AI 摘要生成失敗。'

In [28]:


def main(new_grades_to_write):
    """
    主程式流程：獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.Client(auth=creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.get_worksheet(1) # 取得第二個工作表

        print("--- Google Sheet 連線成功。---")

        if not new_grades_to_write:
            print("沒有提供成績資料，程式結束。")
            return

        # 2. 將新成績寫入 Google Sheet，並對應到正確的欄位
        rows_to_append = []
        for record in new_grades_to_write:
            year, semester, subject, grade = record
            padded_row = [year, semester, subject, grade]
            rows_to_append.append(padded_row)

        ws.append_rows(rows_to_append)

        print("\n--- 成績已成功寫入 Google Sheet。---")

        # . 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades_to_write)

        # 尋找空白列、與上方成績資料空一行
        next_row = len(ws.col_values(1)) + 2

        # 使用 update_cell() 方法更新儲存格，將整個摘要內容放入一個儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI成績建議')
        ws.update_cell(next_row, 3, summary) # 將完整的 summary 放入單一儲存格

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

# Call main with the existing new_grades. The if __name__ == "__main__" part is removed here as it's not needed when calling directly.
main(new_grades)

--- Google Sheet 連線成功。---

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---
呼叫 AI 時發生錯誤：503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
AI 摘要生成失敗。
--------------------------------------------------


In [46]:
import gradio as gr

REQUIRED_COLUMNS = ["學年度", "學期", "科目", "成績"]

# Define the function that will process the grades and generate summary
def process_grades_and_summary(grades_data, clear_sheet):
    sheet_status_messages = []
    ai_summary_text = "AI 摘要生成失敗。" # Default failure message

    # Convert grades_data from Gradio Dataframe format to expected format
    processed_grades = []
    for i, row in enumerate(grades_data):
        # Skip empty rows from Gradio Dataframe
        if not any(str(item).strip() for item in row):
            continue

        try:
            # Expecting row to be [year, semester, subject, grade]
            if len(row) != len(REQUIRED_COLUMNS):
                sheet_status_messages.append(f"錯誤：第 {i+1} 行成績資料欄位數量不正確。預期 {len(REQUIRED_COLUMNS)} 個欄位。")
                continue # Skip this malformed row for further processing

            year = int(row[0])
            semester = int(row[1])
            subject = str(row[2])
            grade = int(row[3])
            processed_grades.append([year, semester, subject, grade])
        except (ValueError, IndexError) as e:
            sheet_status_messages.append(f"錯誤：第 {i+1} 行成績資料格式不正確，請確保學年度、學期和成績是數字。錯誤詳情：{e}")
            continue # Skip this malformed row for further processing

    if not processed_grades:
        return "沒有輸入有效的成績資料。", "沒有成績資料可以生成摘要。"

    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.Client(auth=creds)
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.get_worksheet(1) # 取得第二個工作表
        sheet_status_messages.append("Google Sheet 連線成功。")

        # Clear sheet if checkbox is checked
        if clear_sheet:
            # Get the number of rows in the worksheet that contain values
            num_populated_rows = len(ws.get_all_values())
            # If there's more than just a potential header row, delete data rows
            if num_populated_rows > 1:
                ws.delete_rows(2, num_populated_rows) # Delete rows from the second row to the last populated row
            sheet_status_messages.append("Google Sheet 已清除。")

        # 2. 將新成績寫入 Google Sheet
        rows_to_append = []
        for record in processed_grades:
            rows_to_append.append(record)
        ws.append_rows(rows_to_append)
        sheet_status_messages.append("成績已成功寫入 Google Sheet。")

        # 3. 獲取 AI 摘要
        summary = get_ai_summary(processed_grades)
        ai_summary_text = summary # Update AI summary with actual result

        # 4. 尋找空白列、與上方成績資料空一行並寫入 AI 摘要
        next_row = len(ws.col_values(1)) + 2
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI成績建議')
        ws.update_cell(next_row, 3, summary) # 將完整的 summary 放入單一儲存格

        # Only append success message if AI summary was not a failure
        if ai_summary_text != "AI 摘要生成失敗。":
            sheet_status_messages.append("AI 摘要已成功寫入 Google Sheet。")

    except gspread.exceptions.APIError as e:
        sheet_status_messages.append(f"Google Sheets API 錯誤：{e.response.text}")
        sheet_status_messages.append("請確認：1. 您的服務帳戶金鑰檔案正確且未過期。2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
        ai_summary_text = "AI 摘要生成失敗 (Google Sheet 錯誤)。"
    except Exception as e:
        sheet_status_messages.append(f"發生未預期的錯誤：{e}")
        ai_summary_text = "AI 摘要生成失敗 (程式執行錯誤)。"

    return "\n".join(sheet_status_messages), ai_summary_text


with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的學年度、學期、科目和成績，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Row():
        with gr.Column():
            grade_input = gr.Dataframe(
                headers=REQUIRED_COLUMNS,
                value=[["", "", "", ""]], # Initial empty row for all 4 columns
                type="array",
                row_count=(1, "dynamic"), # Allow dynamic row addition
                col_count=(len(REQUIRED_COLUMNS), "fixed"),
                label="輸入學年度、學期、科目與成績 (點擊最後一行可新增)"
            )
            clear_sheet_checkbox = gr.Checkbox(label="寫入前清除 Google Sheet", value=False)
            submit_button = gr.Button("送出")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    submit_button.click(
        process_grades_and_summary,
        inputs=[grade_input, clear_sheet_checkbox],
        outputs=[sheet_output, summary_output]
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b52e660ad757facf32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
